<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_105_attention_mechanism_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🎯 **Module 105 — Attention Mechanism Analysis** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 15 · Mechanistic Interpretability Workshop (Cohen ML4LLM deep-dives)


# Module 105 — Attention Mechanism Analysis

> M19/M20 built attention from scratch. M104 introduced residual-stream interventions. **This module opens up the attention block itself** as data: Cohen Ch6 (projects 33-41) extracts and analyzes **Q · K · V weight statistics**, **raw vs softmax attention scores**, **the per-head attention-adjustment magnitudes** that contribute back to the residual stream, **KL divergences** between attention patterns and token predictions, **token-frequency × attention** correlations, and the two foundational head-level interventions: **head silencing** and **attention patching for IOI**. By the end you'll know how to read a model's attention map like an EKG.

### What you'll cover
1. The **attention architecture revisited** — Q, K, V, heads, MHA
2. **QKV weight characteristics** — norms, anisotropy, head-rank
3. **QKV activation characteristics** — per-token, per-position
4. **Raw vs softmax attention scores** — the temperature of attention
5. **Attention-adjustment magnitudes** — how much each head writes to residual
6. **KL divergences** between token-prediction distributions vs attention patterns
7. **Laminar RSA** — depth profile of category selectivity
8. **Token-frequency × attention × QK^T** — frequency-bias of heads
9. **Head silencing** — zero a head, measure the damage
10. **Attention patching in IOI** + the 2026 attention-interp stack


## 1 · Attention architecture revisited

A single Transformer attention block (decoder-only):

```
input  h ∈ ℝ^{T × D}
         │
         ▼  LayerNorm
         │
    Q = h · W_Q   ∈ ℝ^{T × D}      reshape → [H heads, T, d_k]
    K = h · W_K   ∈ ℝ^{T × D}      reshape → [H heads, T, d_k]
    V = h · W_V   ∈ ℝ^{T × D}      reshape → [H heads, T, d_v]

per head h:
    scores  = Q_h · K_h^T / √d_k   ∈ ℝ^{T × T}        # the "QK^T" map
    masked  = causal mask
    A_h     = softmax(scores, dim=-1)                 # attention pattern
    out_h   = A_h · V_h            ∈ ℝ^{T × d_v}

concatenate heads → out · W_O ∈ ℝ^{T × D}             # write back to residual stream
```

The **four matrices per layer**: `W_Q, W_K, W_V, W_O ∈ ℝ^{D × D}` (or `D × H · d_k`). Cohen Ch6 analyzes all four plus the per-head activations.

> 🧠 **Mental model.** Q is the *query* ("what am I looking for?"), K is the *key* ("what do I offer?"), V is the *value* ("what do I deliver?"). Attention reads a weighted sum of values, where weights = softmax(Q · K^T). Heads run this circuit in parallel with independent Q/K/V/O matrices but share residual reads/writes.


## 2 · QKV weight characteristics

Cohen proj 33 looks at the **static weight matrices** before training is done. Three statistics per head per layer:

| Stat | What it tells you |
|---|---|
| **Frobenius norm** | Overall "strength" of the head |
| **Singular value spectrum** | Effective rank of the head |
| **Row-norm distribution** | Which input dims this head listens to |
| **Q-K asymmetry** | Whether Q and K matrices are correlated (cause-effect) or random |
| **Init-residual ratio** | How far the trained weights moved from initialization |

Typical findings (Llama-3-8B / GPT-2-large / Mistral-7B):
- **Most heads have low Frobenius norm** — they barely matter (the "dead heads" of 2024 head-pruning literature)
- **A small subset (5-15% of heads) has 3-10× higher norm** — these are the workhorse heads
- **Effective rank per Q matrix ≈ 8-32** out of `d_k = 64-128` — heads operate in low-rank subspaces

Practical: pruning the bottom-50% of heads by Frobenius norm often costs <2 perplexity points. This is the basis of **Pruning Heads is Not All You Need** (Voita 2019 → Michel 2019 → SliceGPT 2024 → Wanda 2023 → Sheared-LLaMA 2024) — head pruning is among the most-studied LLM-compression techniques (M90 callback).


## 3 · QKV activation characteristics

Cohen proj 34 looks at the **runtime activations** — `Q_h(token)`, `K_h(token)`, `V_h(token)` for actual prompts:

```
Per token, per head:
    ||Q_h||, ||K_h||, ||V_h||             # vector magnitudes
    angle(Q_h, K_h)                       # are they aligned?
    sparsity of |Q_h|, |K_h|              # how peaked?
```

Three discoveries (replicate Cohen's notebook on GPT-2 or Llama-3):
- **`||Q||` ≈ `||K||`** for most heads (init regularizes this)
- **Some heads have huge `||V||` outliers** — the "value-outlier features" of Dettmers 2022 / Sun & Dettmers 2024 → motivates **outlier-aware INT8 / FP8 quantization** (M90 callback)
- **Q and K are usually nearly orthogonal until "attention day"** — the head fires only when a specific Q-K alignment is met (the "attention key-query lock")

> 🧨 **Why outlier features matter for production.** Sun & Dettmers' 2024 paper showed that a tiny number of channels (~0.5%) carry most of `||V||`'s magnitude. INT8 quantization without isolating these channels destroys accuracy. **LLM.int8()** and **SmoothQuant** were designed around this finding.


## 4 · Raw vs softmax attention

Cohen proj 35: **raw scores** `Q · K^T / √d_k` vs **softmax attention** `A = softmax(scores)`.

Raw scores cover a wide range (e.g., -20 to +20 for an 8B model). The softmax compresses them sharply:

```
score = 10  →  exp(10) = 22026
score = 9   →  exp(9)  =  8103       → ~3× less weight
score = 0   →  exp(0)  =     1       → ~22000× less weight
```

Two takeaways:
1. **Softmax is winner-take-most.** A single high score gets ~50-80% of the attention mass; the next 2-3 split most of the remainder; the rest get rounding error.
2. **The "spread" of attention is per-head.** Some heads have one peak (focused) — induction heads, copy-and-paste heads. Other heads spread across many positions (diffuse) — the "everywhere" heads that produce a context summary.

**Attention entropy** `H(A_h(t, :))` measures this. Low entropy = focused; high entropy = diffuse. The distribution of head-entropies is itself a fingerprint of the model:

```
Focused heads (entropy < 1.0)     |  ~ 20% of heads — induction, BOS, name-mover
Mid heads (entropy 1.0 - 3.0)     |  ~ 50% of heads — semantic, syntactic
Diffuse heads (entropy > 3.0)     |  ~ 30% of heads — summary, context
```

This is exactly what Anthropic's 2022 "in-context learning circuits" paper measured to find **induction heads**.


## 5 · Attention-adjustment magnitudes — how much each head writes

Cohen proj 36 measures **per-head contribution to the residual stream**:

```
out_h = A_h · V_h        ∈ ℝ^{T × d_v}        # per-head output
contrib_h = out_h · W_O[h-slice]             # projected back to residual D-dim
||contrib_h||                                # magnitude of this head's write
```

**Findings:**
- **Magnitude varies 10-100× across heads** within a single layer
- **Top-5 heads per layer carry 50-70% of total magnitude** — a Pareto-style concentration
- **Small heads ("low-volume") still matter for specific tasks** — name-movers in IOI are not the highest-volume heads, but the most causally critical

This connects to the 2024 paper *"The Unreasonable Effectiveness of a Few Attention Heads"* — most heads can be ablated; a small critical subset cannot.

> 📐 **Production implication for LoRA targeting (M28 callback).** LoRA fine-tuning typically updates `W_Q, W_V` (not `W_K, W_O`). Why? `W_Q, W_V` are where most of the *learnable update* lives in fine-tuning; updates to `W_K, W_O` have lower effective rank. Cohen's head-magnitude analysis is the experimental basis.


## 6 · KL divergence — attention vs token-prediction distributions

Cohen proj 37 connects two distributions at the same layer:
1. **Attention pattern** `A_h(t, :)` — the head's distribution over previous positions
2. **Token-prediction distribution** `p_t = softmax(W_out · h^{(l)}_t)` — what would the model predict if you read the residual stream now?

```
KL_h^{(l)}(t) = KL(A_h^{(l)}(t, :)  ‖  p_t)
```

Surprisingly, for many heads this is **highly informative** — head attention patterns *correlate* with the token-prediction distribution:
- Heads with low KL "agree" with the current prediction — they reinforce
- Heads with high KL "disagree" — they bring contrarian evidence

**Application:** identifying heads that *steer* a prediction (low KL but high `||contrib_h||`) vs heads that act as *negative space* (high KL — these are the ones to ablate when steering against a bias).


## 7 · Laminar RSA — depth profile of category selectivity

Cohen proj 38 generalizes the M102 RSA to layers: compute representational-similarity matrices at *every* layer, then **track how category selectivity changes with depth**.

```
Stimuli: 200 words across {animals, vehicles, emotions, places}
For each layer l:
    H_l = mean over context of hidden states for each stimulus
    S_l = N × N cosine matrix
    selectivity_l = (S_in − S_out) / (S_in + S_out)
```

Llama-3-8B profile across 32 layers:

```
selectivity
   ↑     emotions
       ╱──╲
        ╲   ╲___ semantics
              ╲
                ╲___  tokens (early peak)
   ────────────────────────────────► layer depth
```

This is the **laminar profile** — like a brain's cortical column. Different "kinds" of category selectivity peak at different depths. Production: pick the layer whose selectivity profile matches your task.

> 🧠 **Neuroscience connection.** Cohen's neuroscience background shows here — "laminar" comes from cortical layer analysis. RSA was *originally* invented for fMRI / electrode recordings. LLMs and brains both turn out to have layer-specific category specialization that RSA can diagnose.


## 8 · Token frequency × attention × QK^T

Cohen proj 39 asks: **does attention correlate with token frequency?** Plot mean attention received by each token vs its training-corpus frequency:

```
Common tokens (the, of, ., \n) — receive disproportionate attention
Rare tokens (proper nouns, code) — receive narrow but high-weight attention from specific heads
```

Two phenomena emerge:
1. **The "BOS-sink" / "attention sink" effect** (Xiao 2023): the *first* token (BOS or any) receives 30-60% of attention from many heads, regardless of content. Removing it breaks the model. This is what **Streaming-LLM** keeps in cache.
2. **Frequency-aware heads**: some heads have `QK^T` matrices that explicitly favor common-vs-rare tokens. These can be visualized as the per-head correlation `corr(token-frequency, attention-received)`.

**Practical consequence (M86 / M53 RAG callback).** Production RAG / agents using **KV cache compression** (H2O, StreamingLLM, MInference) all rely on the attention-sink + frequency-bias finding to decide *which keys to evict* from cache. Without this insight, KV-cache compression would be lossy on every layer.


## 9 · Head silencing — zero a head, measure damage

Cohen proj 40 is the **first attention-head intervention**: ablate one head at a time, measure perplexity / accuracy on a held-out test set.

```
For each (layer, head) pair:
    set A_h(l) ← uniform(1/T)   OR   zero out the head's contribution
    Run model end-to-end.
    Measure ΔPPL or Δaccuracy on a benchmark.
```

You discover three classes of heads (consistent across architectures):
- **Critical heads** (~5-10% of all heads) — ablating one causes >5% PPL increase
- **Replaceable heads** (~50% of all heads) — small impact; redundant with others
- **Dead heads** (~20-40% of all heads) — zero impact; sometimes have near-zero `||W||` too

**SliceGPT** (Microsoft 2024) and **Sheared-LLaMA** (Princeton 2024) operationalize this: ablate the bottom-30% of heads (by impact-magnitude product) without significant quality loss → ~30% smaller model, ~30% faster inference.

> 🛠️ **Head silencing as a production tool.** Once you have an importance ranking, you can ship a **head-sparse** version for edge deployment (M90 callback). Llama-3.2-3B is rumored to use a head-pruned ancestor of Llama-3-8B; the relationship between the two embedding spaces strongly suggests head silencing was the compression mechanism.


## 10 · Attention patching in IOI + the 2026 stack

Cohen proj 41 — the attention-level version of M104's residual-stream patching. **Same IOI prompt pair** (clean vs corrupted), but instead of patching residual stream, patch the *attention pattern* at a specific head:

```
Clean:     "When Mary and John went, John gave a drink to" → "Mary"
Corrupted: "When John and Mary went, Mary gave a drink to" → "John"

Patch A_h^{(l)} at (l, h):
    Run corrupted, but force A_h^{(l)} = the clean-prompt attention pattern at the same head.
    Measure: did "Mary" become higher prob?
```

This **isolates the causal contribution of one head**. Run for all `(l, h)` pairs → discover the **IOI circuit**:

```
Layer 9: "S-inhibition heads" suppress the subject (John)
Layer 10-11: "Name-mover heads" copy the object (Mary)
Layer 11: "Backup name-mover" provides redundancy
```

This 3-stage circuit, discovered by Wang 2023, is the **canonical mech-interp result**. It's been replicated in GPT-2, Pythia, Llama-2/3, Gemma, Mistral, Qwen — the *same circuit pattern* generalizes across models.

**The 2026 attention-interp toolkit:**

| Library | Maintainer | Use |
|---|---|---|
| **TransformerLens** | Neel Nanda | The reference mech-interp library; activation/head/layer hooks |
| **nnsight** | NDIF (Northeastern) | Distributed-friendly; works on Llama-3-405B |
| **Pyvene** | Stanford | Compositional interventions |
| **SAE Lens** | Joseph Bloom | Sparse Autoencoder training + analysis |
| **ActAdd / steering** | Anthropic | Steering vectors |
| **CircuitsVis** | Anthropic | Interactive circuit visualization |
| **Inspect / TextGrad / DeepSeek-V3 Attention** | Various | Mid-2026 production interp tools |

> 🎓 **The Cohen-Ch6 capstone.** Attention is **not a black box** — every head has a measurable role you can describe with `(layer, head, function-class, contribution-magnitude, KL-with-prediction)`. The mech-interp craft is making this description precise and *causally testable*. After M104 + M105 you can read most papers from Anthropic / OpenAI Interp / DeepMind ROME-MEMIT / Apollo Research and understand what's being claimed.

## ✅ Recap

- **Attention block** = `W_Q, W_K, W_V, W_O` per layer × H heads; Cohen Ch6 analyzes all four + activations.
- **QKV weight stats** reveal a few workhorse heads + many low-norm "dead" heads — head-pruning enabler.
- **QKV activations** show outlier features in `||V||` that motivate INT8/FP8 quantization (LLM.int8, SmoothQuant).
- **Raw vs softmax** — winner-take-most; attention entropy fingerprints heads (focused, mid, diffuse).
- **Attention-adjustment magnitudes** — Pareto: top-5 heads carry 50-70% of magnitude in a layer.
- **KL(attention ‖ prediction)** identifies heads that *reinforce* vs *steer against* the current prediction.
- **Laminar RSA** = depth profile of category selectivity; tokens early, semantics middle, sentiment late.
- **Token-frequency × attention** reveals the **attention-sink** (BOS) + frequency-bias — basis of StreamingLLM / KV-cache compression.
- **Head silencing** finds critical (~5-10%), replaceable (~50%), dead (~20-40%) heads — basis of SliceGPT, Sheared-LLaMA.
- **Attention patching in IOI** finds the **S-inhibition + name-mover + backup name-mover** 3-stage circuit — Wang 2023 canonical result.
- **2026 toolkit**: TransformerLens, nnsight, Pyvene, SAE Lens, ActAdd, CircuitsVis.

Next: **M106 — MLP Probing + Steering** (Cohen Ch7) — the final module.
